# Select images for analysis

In [ ]:
input_dirpath = Path(input())

In [ ]:
# Create CSV file with all of the image names
imgnames = [f.name for f in input_dirpath.glob('*.ome.tif')]
imgnames.sort()
df = pd.DataFrame({'image name': imgnames})
df.head()

In [ ]:
# Obtain img info from naming scheme (requires specific naming convention)
namesplits = df['image name'].str.split('.ome.tif').str[0].str.split('_')
df['experiment'] = namesplits.str[0]
df['wellID'] = namesplits.str[2]
df['scene'] = namesplits.str[3]
df['img idx'] = np.arange(0, len(df))
df.head()

In [ ]:
# Requests user input for path of a 'Well Conditions' csv file containing treatment information for each well
wellcond_df_path = Path(input())

# Displays the well conditions dataframe
wellcond_df = pd.read_csv(wellcond_df_path)
wellcond_df.head()

In [ ]:
# Fills in information about the current img list from the well conditions dataframe
num_rows_premerge = len(df)
df = pd.merge(df, wellcond_df, how='left', on=['experiment', 'wellID'], suffixes=['', '_y'], validate='many_to_one')
num_rows_postmerge = len(df)
assert num_rows_premerge==num_rows_postmerge
df

In [ ]:
# Counts the number of images in each well
tables_dirpath = utils.get_proc_dirpath(input_dirpath) / dn.tables_dirname
tablename = 'img_counts.csv'
counts = df.groupby(['wellID'])['image name'].count()
counts = counts.rename('num imgs')
counts

In [ ]:
# Saves well counts to a separate dataframe
counts.to_csv(tables_dirpath / tablename, index=False)

In [ ]:
# Select which tx to analyze
tx_to_analyze = ['281', '283']

# Find images corresponding to the tx selected above
conds = [(df['tx']==tx) for tx in tx_to_analyze]
vals = [True, True]
selected = np.select(conds, vals)
selected_df = df[selection]
selected_df

In [ ]:
# Decide how many images to anayze per tx
num_selected_imgs = 20

# Randomly select images to analyze
selected_df = selected_df.groupby('wellID').sample(n=num_selected_imgs, random_state=1)
selected_df

In [ ]:
# Save names of selected images to a table
tablename = 'selected_imgs.csv'
utils.safe_save_csv(selected_df, tables_dirpath / tablename)